# DA6401 Assignment 3 - Transformer NMT
This notebook is set up for a long unattended Kaggle run.

Notes:
- W&B is hardcoded below and resumes the same run id across Kaggle restarts.
- Google Drive upload uses `GDRIVE_CREDENTIALS` if you add that Kaggle secret; otherwise training still completes and saves locally.
- Uploaded checkpoint names are versioned with the run id so they do not overwrite your older `.pt` file.


In [ ]:
%%capture
import os
import subprocess
import sys

os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

pkgs = [
    'wandb', 'datasets', 'evaluate', 'sacrebleu',
    'gdown', 'pydrive2', 'spacy'
]
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p], check=True)

for model_name in ['de_core_news_sm', 'en_core_web_sm']:
    try:
        __import__(model_name)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'spacy', 'download', model_name], check=True)


In [ ]:
import os
import wandb

WANDB_API_KEY = "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"
WANDB_PROJECT = 'da6401-a3-submission'

os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['WANDB_PROJECT'] = WANDB_PROJECT

RUN_ID_FILE = '/kaggle/working/wandb_run_id.txt'
wandb.login(key=WANDB_API_KEY, relogin=True)

if os.path.exists(RUN_ID_FILE):
    with open(RUN_ID_FILE) as f:
        WANDB_RUN_ID = f.read().strip()
    print(f'Resuming existing W&B run: {WANDB_RUN_ID}')
else:
    WANDB_RUN_ID = wandb.util.generate_id()
    with open(RUN_ID_FILE, 'w') as f:
        f.write(WANDB_RUN_ID)
    print(f'New W&B run ID generated and saved: {WANDB_RUN_ID}')

os.environ['WANDB_RUN_ID'] = WANDB_RUN_ID
print(f'W&B logged in. Project: {WANDB_PROJECT}')


In [ ]:
import json
import os
import tempfile
from datetime import datetime

GDRIVE_FOLDER_ID = '1U-xufTFYBDh8MSl6wNyBKx_2P_BAdM0m'
os.environ['GDRIVE_FOLDER_ID'] = GDRIVE_FOLDER_ID
os.environ['GDRIVE_UPLOAD_PREFIX'] = f'da6401_a3_{WANDB_RUN_ID}'


def upload_to_gdrive(local_path: str, gdrive_folder_id: str, remote_name: str | None = None):
    """Upload a local file to Google Drive with a run-versioned name."""
    try:
        from pydrive2.auth import GoogleAuth
        from pydrive2.drive import GoogleDrive

        creds_json = os.environ.get('GDRIVE_CREDENTIALS', '')
        if not creds_json:
            print('GDRIVE_CREDENTIALS not set - skipping Drive upload (checkpoint saved locally).')
            return None

        with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
            f.write(creds_json)
            creds_path = f.name

        gauth = GoogleAuth()
        gauth.settings['client_config_backend'] = 'service'
        gauth.settings['service_config'] = {'client_json_file_path': creds_path}
        gauth.ServiceAuth()
        drive = GoogleDrive(gauth)

        basename = os.path.basename(local_path)
        prefix = os.environ.get('GDRIVE_UPLOAD_PREFIX', f'da6401_a3_{WANDB_RUN_ID}')
        timestamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
        remote_title = remote_name or f'{prefix}_{timestamp}_{basename}'

        file_obj = drive.CreateFile({'title': remote_title, 'parents': [{'id': gdrive_folder_id}]})
        file_obj.SetContentFile(local_path)
        file_obj.Upload()
        print(f'Uploaded {basename} to Google Drive as {remote_title}')
        return file_obj.get('id')
    except Exception as e:
        print(f'Drive upload failed (non-fatal): {e}')
        print('Checkpoint is still saved locally at:', local_path)
        return None


In [ ]:
# Write all source files inline so this notebook is fully self-contained

import os
os.makedirs('/kaggle/working/da6401_a3', exist_ok=True)
os.chdir('/kaggle/working/da6401_a3')

MODEL_PY = r'''"""
model.py — Transformer Architecture
DA6401 Assignment 3: "Attention Is All You Need"

AUTOGRADER CONTRACT (DO NOT MODIFY SIGNATURES):
  ┌─────────────────────────────────────────────────────────────────┐
  │  scaled_dot_product_attention(Q, K, V, mask) → (out, weights)  │
  │  MultiHeadAttention.forward(q, k, v, mask)   → Tensor          │
  │  PositionalEncoding.forward(x)               → Tensor          │
  │  make_src_mask(src, pad_idx)                 → BoolTensor      │
  │  make_tgt_mask(tgt, pad_idx)                 → BoolTensor      │
  │  Transformer.encode(src, src_mask)           → Tensor          │
  │  Transformer.decode(memory,src_m,tgt,tgt_m)  → Tensor          │
  └─────────────────────────────────────────────────────────────────┘
"""

import copy
import math
import os
from typing import Callable, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

# ══════════════════════════════════════════════════════════════════════
#  STANDALONE ATTENTION FUNCTION
# ══════════════════════════════════════════════════════════════════════


def scaled_dot_product_attention(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    mask: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Compute Scaled Dot-Product Attention.
        Attention(Q, K, V) = softmax( Q·Kᵀ / √dₖ ) · V

    Args:
        Q    : Query tensor,  shape (..., seq_q, d_k)
        K    : Key tensor,    shape (..., seq_k, d_k)
        V    : Value tensor,  shape (..., seq_k, d_v)
        mask : Optional Boolean mask, broadcastable to (..., seq_q, seq_k).
               Positions where mask is True are MASKED OUT (set to -inf).

    Returns:
        output : shape (..., seq_q, d_v)
        attn_w : shape (..., seq_q, seq_k)  — sums to 1 along last dim
    """
    d_k = Q.size(-1)
    use_scaling = getattr(scaled_dot_product_attention, "use_scaling", True)
    scores = torch.matmul(Q, K.transpose(-2, -1))
    if use_scaling:
        scores = scores / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float("-inf"))
    attn_w = F.softmax(scores, dim=-1)
    # Replace NaN from all-masked rows (softmax of all -inf) with 0
    attn_w = torch.nan_to_num(attn_w, nan=0.0)
    output = torch.matmul(attn_w, V)
    return output, attn_w


# ══════════════════════════════════════════════════════════════════════
#  MASK HELPERS
# ══════════════════════════════════════════════════════════════════════


def make_src_mask(
    src: torch.Tensor,
    pad_idx: int = 1,
) -> torch.Tensor:
    """
    Build a padding mask for the encoder.

    Returns:
        Boolean mask, shape [batch, 1, 1, src_len]
        True  → PAD token (masked out)
        False → real token
    """
    return (src == pad_idx).unsqueeze(1).unsqueeze(2)


def make_tgt_mask(
    tgt: torch.Tensor,
    pad_idx: int = 1,
) -> torch.Tensor:
    """
    Build a combined padding + causal mask for the decoder.

    Returns:
        Boolean mask, shape [batch, 1, tgt_len, tgt_len]
        True → masked out (PAD or future position)
    """
    batch_size, tgt_len = tgt.shape
    pad_mask = (tgt == pad_idx).unsqueeze(1).unsqueeze(2)
    causal_mask = torch.triu(
        torch.ones(tgt_len, tgt_len, device=tgt.device, dtype=torch.bool),
        diagonal=1,
    )
    return pad_mask | causal_mask.unsqueeze(0).unsqueeze(0).expand(
        batch_size, -1, -1, -1
    )


def _detokenize_text(text: str) -> str:
    replacements = {
        " .": ".",
        " ,": ",",
        " !": "!",
        " ?": "?",
        " :": ":",
        " ;": ";",
        " n't": "n't",
        " 's": "'s",
        " 're": "'re",
        " 've": "'ve",
        " 'm": "'m",
        " 'll": "'ll",
        " 'd": "'d",
        "( ": "(",
        " )": ")",
    }
    for src, dst in replacements.items():
        text = text.replace(src, dst)
    return text.strip()


# ══════════════════════════════════════════════════════════════════════
#  MULTI-HEAD ATTENTION
# ══════════════════════════════════════════════════════════════════════


class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention — torch.nn.MultiheadAttention is NOT used.
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        dropout: float = 0.1,
        use_scaling: bool = True,
    ) -> None:
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.use_scaling = use_scaling

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.attn_weights: Optional[torch.Tensor] = None

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        B, L, _ = x.shape
        return x.view(B, L, self.num_heads, self.d_k).transpose(1, 2)

    def _combine_heads(self, x: torch.Tensor) -> torch.Tensor:
        B, _, L, _ = x.shape
        return x.transpose(1, 2).contiguous().view(B, L, self.d_model)

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        Q = self._split_heads(self.W_q(query))
        K = self._split_heads(self.W_k(key))
        V = self._split_heads(self.W_v(value))

        original_scaling = getattr(scaled_dot_product_attention, "use_scaling", True)
        scaled_dot_product_attention.use_scaling = self.use_scaling
        try:
            attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        finally:
            scaled_dot_product_attention.use_scaling = original_scaling
        self.attn_weights = attn_weights  # stored for visualisation

        return self.W_o(self._combine_heads(attn_output))


# ══════════════════════════════════════════════════════════════════════
#  POSITIONAL ENCODING
# ══════════════════════════════════════════════════════════════════════


class PositionalEncoding(nn.Module):
    """
    Sinusoidal Positional Encoding — pe registered as a non-trainable buffer.
    """

    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000) -> None:
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)


class LearnedPositionalEncoding(nn.Module):
    """Learned positional embeddings for ablation experiments."""

    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000) -> None:
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        self.pe = nn.Embedding(max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        return self.dropout(x + self.pe(positions))


# ══════════════════════════════════════════════════════════════════════
#  FEED-FORWARD NETWORK
# ══════════════════════════════════════════════════════════════════════


class PositionwiseFeedForward(nn.Module):
    """FFN(x) = max(0, xW₁+b₁)W₂+b₂"""

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear2(self.dropout(F.relu(self.linear1(x))))


# ══════════════════════════════════════════════════════════════════════
#  ENCODER LAYER
# ══════════════════════════════════════════════════════════════════════


class EncoderLayer(nn.Module):
    """Self-Attn → Add&Norm → FFN → Add&Norm  (Post-LN)"""

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        dropout: float = 0.1,
        use_scaling: bool = True,
    ) -> None:
        super().__init__()
        self.self_attn = MultiHeadAttention(
            d_model, num_heads, dropout, use_scaling=use_scaling
        )
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, src_mask: torch.Tensor) -> torch.Tensor:
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, src_mask)))
        x = self.norm2(x + self.dropout(self.feed_forward(x)))
        return x


# ══════════════════════════════════════════════════════════════════════
#  DECODER LAYER
# ══════════════════════════════════════════════════════════════════════


class DecoderLayer(nn.Module):
    """Masked-Self-Attn → Add&Norm → Cross-Attn → Add&Norm → FFN → Add&Norm"""

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        dropout: float = 0.1,
        use_scaling: bool = True,
    ) -> None:
        super().__init__()
        self.self_attn = MultiHeadAttention(
            d_model, num_heads, dropout, use_scaling=use_scaling
        )
        self.cross_attn = MultiHeadAttention(
            d_model, num_heads, dropout, use_scaling=use_scaling
        )
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        memory: torch.Tensor,
        src_mask: torch.Tensor,
        tgt_mask: torch.Tensor,
    ) -> torch.Tensor:
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, memory, memory, src_mask)))
        x = self.norm3(x + self.dropout(self.feed_forward(x)))
        return x


# ══════════════════════════════════════════════════════════════════════
#  ENCODER & DECODER STACKS
# ══════════════════════════════════════════════════════════════════════


class Encoder(nn.Module):
    """Stack of N identical EncoderLayers + final LayerNorm."""

    def __init__(self, layer: EncoderLayer, N: int) -> None:
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])
        self.norm = nn.LayerNorm(layer.norm1.normalized_shape[0])

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)


class Decoder(nn.Module):
    """Stack of N identical DecoderLayers + final LayerNorm."""

    def __init__(self, layer: DecoderLayer, N: int) -> None:
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])
        self.norm = nn.LayerNorm(layer.norm1.normalized_shape[0])

    def forward(
        self,
        x: torch.Tensor,
        memory: torch.Tensor,
        src_mask: torch.Tensor,
        tgt_mask: torch.Tensor,
    ) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)


# ══════════════════════════════════════════════════════════════════════
#  FULL TRANSFORMER
# ══════════════════════════════════════════════════════════════════════


class Transformer(nn.Module):
    """
    Full Encoder-Decoder Transformer for sequence-to-sequence tasks.

    src_vocab_size and tgt_vocab_size have defaults so the autograder
    can call Transformer() with no arguments.
    """

    def __init__(
        self,
        src_vocab_size: int = 8000,
        tgt_vocab_size: int = 6000,
        d_model: int = 256,
        N: int = 3,
        num_heads: int = 8,
        d_ff: int = 512,
        dropout: float = 0.1,
        pad_idx: int = 1,
        src_itos: Optional[dict] = None,
        tgt_itos: Optional[dict] = None,
        tgt_stoi: Optional[dict] = None,
        src_stoi: Optional[dict] = None,
        src_tokenizer: Optional[Callable] = None,
        checkpoint_path: str = None,
        max_len: int = 100,
        use_learned_positional_encoding: bool = False,
        attention_use_scaling: bool = True,
    ) -> None:
        super().__init__()
        self.src_vocab_size = src_vocab_size
        self.tgt_vocab_size = tgt_vocab_size
        self.d_model = d_model
        self.N = N
        self.num_heads = num_heads
        self.d_ff = d_ff
        self.dropout_rate = dropout
        self.pad_idx = pad_idx
        self.max_len = max_len
        self.src_itos = src_itos or {}
        self.tgt_itos = tgt_itos or {}
        self.tgt_stoi = tgt_stoi or {}
        self.src_stoi = src_stoi or {}
        self.src_tokenizer = src_tokenizer
        self.use_learned_positional_encoding = use_learned_positional_encoding
        self.attention_use_scaling = attention_use_scaling

        self.src_embed = nn.Embedding(src_vocab_size, d_model, padding_idx=pad_idx)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model, padding_idx=pad_idx)
        pos_cls = (
            LearnedPositionalEncoding
            if use_learned_positional_encoding
            else PositionalEncoding
        )
        self.src_positional_encoding = pos_cls(d_model, dropout, max_len=max_len)
        self.tgt_positional_encoding = pos_cls(d_model, dropout, max_len=max_len)

        enc_layer = EncoderLayer(
            d_model, num_heads, d_ff, dropout, use_scaling=attention_use_scaling
        )
        dec_layer = DecoderLayer(
            d_model, num_heads, d_ff, dropout, use_scaling=attention_use_scaling
        )
        self.encoder = Encoder(enc_layer, N)
        self.decoder = Decoder(dec_layer, N)
        self.output_linear = nn.Linear(d_model, tgt_vocab_size, bias=False)
        self.output_linear.weight = self.tgt_embed.weight  # weight tying

        self._reset_parameters()

        should_autoload_default_checkpoint = (
            checkpoint_path is None
            and not self.src_stoi
            and not self.tgt_stoi
            and src_vocab_size == 8000
            and tgt_vocab_size == 6000
            and d_model == 256
            and N == 3
            and num_heads == 8
            and d_ff == 512
        )
        if should_autoload_default_checkpoint:
            checkpoint_path = "best_model.pt"

        if checkpoint_path is not None:
            import gdown

            if not os.path.exists(checkpoint_path):
                gdown.download(
                    id="1OWO62R9qm4RImEKhW6RMBFCjojYLNvQn",
                    output=checkpoint_path,
                    quiet=False,
                )
            if os.path.exists(checkpoint_path):
                state = torch.load(checkpoint_path, map_location="cpu")
                if "model_state_dict" in state:
                    vocab_metadata = state.get("vocab_metadata", {})
                    self.src_stoi = vocab_metadata.get("src_stoi", self.src_stoi)
                    self.tgt_stoi = vocab_metadata.get("tgt_stoi", self.tgt_stoi)
                    self.src_itos = vocab_metadata.get("src_itos", self.src_itos)
                    self.tgt_itos = vocab_metadata.get("tgt_itos", self.tgt_itos)
                    state = state["model_state_dict"]
                self.load_state_dict(state, strict=False)

    def _reset_parameters(self) -> None:
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    # ── AUTOGRADER HOOKS ─────────────────────────────────────────────

    def encode(self, src: torch.Tensor, src_mask: torch.Tensor) -> torch.Tensor:
        x = self.src_positional_encoding(self.src_embed(src) * math.sqrt(self.d_model))
        return self.encoder(x, src_mask)

    def decode(
        self,
        memory: torch.Tensor,
        src_mask: torch.Tensor,
        tgt: torch.Tensor,
        tgt_mask: torch.Tensor,
    ) -> torch.Tensor:
        x = self.tgt_positional_encoding(self.tgt_embed(tgt) * math.sqrt(self.d_model))
        return self.output_linear(self.decoder(x, memory, src_mask, tgt_mask))

    def forward(
        self,
        src: torch.Tensor,
        tgt: torch.Tensor,
        src_mask: torch.Tensor,
        tgt_mask: torch.Tensor,
    ) -> torch.Tensor:
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)

    # ── INFERENCE ────────────────────────────────────────────────────

    def infer(self, src_sentence: str) -> str:
        """
        Translate a German sentence to English using greedy decoding.
        Must remain fully offline during autograder evaluation.
        """
        device = next(self.parameters()).device
        src_stoi = (
            self.src_stoi
            if self.src_stoi
            else {"<unk>": 0, "<pad>": 1, "<sos>": 2, "<eos>": 3}
        )
        tgt_stoi = (
            self.tgt_stoi
            if self.tgt_stoi
            else {"<unk>": 0, "<pad>": 1, "<sos>": 2, "<eos>": 3}
        )
        tgt_itos = (
            self.tgt_itos
            if self.tgt_itos
            else {idx: tok for tok, idx in tgt_stoi.items()}
        )

        if self.src_tokenizer is not None:
            if hasattr(self.src_tokenizer, "tokenizer"):
                src_tokens = [
                    tok.text.lower()
                    for tok in self.src_tokenizer.tokenizer(src_sentence)
                ]
            else:
                src_tokens = [
                    tok.text.lower() for tok in self.src_tokenizer(src_sentence)
                ]
        else:
            src_tokens = src_sentence.lower().strip().split()

        tokens = ["<sos>"] + src_tokens[: self.max_len - 2] + ["<eos>"]
        unk = src_stoi.get("<unk>", 0)
        src_ids = [src_stoi.get(t, unk) for t in tokens]
        src_t = torch.tensor(src_ids, dtype=torch.long, device=device).unsqueeze(0)
        src_mask = make_src_mask(src_t, pad_idx=self.pad_idx)

        sos_idx = tgt_stoi.get("<sos>", 2)
        eos_idx = tgt_stoi.get("<eos>", 3)

        self.eval()
        with torch.no_grad():
            memory = self.encode(src_t, src_mask)
            ys = torch.tensor([[sos_idx]], dtype=torch.long, device=device)
            for _ in range(self.max_len - 1):
                tgt_mask = make_tgt_mask(ys, pad_idx=self.pad_idx)
                out = self.decode(memory, src_mask, ys, tgt_mask)
                nxt = int(out[:, -1, :].argmax(dim=-1).item())
                ys = torch.cat(
                    [ys, torch.tensor([[nxt]], dtype=torch.long, device=device)],
                    dim=1,
                )
                if nxt == eos_idx:
                    break

        skip = {sos_idx, eos_idx, tgt_stoi.get("<pad>", 1)}
        return _detokenize_text(
            " ".join(
                tgt_itos[i]
                for i in ys.squeeze(0).tolist()
                if i not in skip and i in tgt_itos
            )
        )
'''
LR_PY = r'''"""
Noam Learning Rate Scheduler
Formula: lrate = d_model^(-0.5) * min(step^(-0.5), step * warmup_steps^(-1.5))
"""

import torch
import torch.optim as optim
from torch.optim.lr_scheduler import LRScheduler


class NoamScheduler(LRScheduler):
    def __init__(
        self,
        optimizer: optim.Optimizer,
        d_model: int,
        warmup_steps: int,
        last_epoch: int = -1,
    ) -> None:
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        super().__init__(optimizer, last_epoch=last_epoch)

    def _get_lr_scale(self) -> float:
        step = max(1, self.last_epoch + 1)
        return (self.d_model ** -0.5) * min(
            step ** -0.5,
            step * (self.warmup_steps ** -1.5),
        )

    def get_lr(self) -> list[float]:
        scale = self._get_lr_scale()
        return [base_lr * scale for base_lr in self.base_lrs]


def get_lr_history(
    d_model: int,
    warmup_steps: int,
    total_steps: int,
) -> list[float]:
    """
    Simulate the LR trajectory of NoamScheduler for `total_steps` steps.

    Args:
        d_model      (int): Model dimensionality.
        warmup_steps (int): Warm-up steps.
        total_steps  (int): Number of steps to simulate.

    Returns:
        list[float]: LR value at each step (length == total_steps).
    """
    dummy_model = torch.nn.Linear(1, 1)
    optimizer = optim.Adam(dummy_model.parameters(), lr=1.0)
    scheduler = NoamScheduler(optimizer, d_model=d_model, warmup_steps=warmup_steps)

    history = []
    for _ in range(total_steps):
        history.append(optimizer.param_groups[0]["lr"])
        optimizer.step()
        scheduler.step()

    return history


if __name__ == "__main__":
    import matplotlib.pyplot as plt

    D_MODEL = 512
    WARMUP_STEPS = 4000
    TOTAL_STEPS = 20_000

    lrs = get_lr_history(D_MODEL, WARMUP_STEPS, TOTAL_STEPS)

    plt.figure(figsize=(9, 4))
    plt.plot(lrs)
    plt.axvline(WARMUP_STEPS, color="red", linestyle="--", label=f"warmup={WARMUP_STEPS}")
    plt.xlabel("Step")
    plt.ylabel("Learning Rate")
    plt.title(f"Noam LR Schedule  (d_model={D_MODEL})")
    plt.legend()
    plt.tight_layout()
    plt.show()
'''
DATASET_PY = r'''"""
dataset.py - Multi30k data pipeline for DA6401 Assignment 3.
"""

import subprocess
import sys
from collections import Counter
from typing import Iterable, List, Sequence, Tuple

import torch
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset


SPECIAL_TOKENS = ["<unk>", "<pad>", "<sos>", "<eos>"]


def _ensure_spacy_model(model_name: str) -> None:
    try:
        __import__(model_name)
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "spacy", "download", model_name],
            check=True,
        )


def load_spacy_models():
    _ensure_spacy_model("de_core_news_sm")
    _ensure_spacy_model("en_core_web_sm")
    import de_core_news_sm
    import en_core_web_sm

    return de_core_news_sm.load(), en_core_web_sm.load()


def tokenize_de(text: str, spacy_de=None) -> List[str]:
    if spacy_de is None:
        spacy_de, _ = load_spacy_models()
    return [tok.text.lower() for tok in spacy_de(text)]


def tokenize_en(text: str, spacy_en=None) -> List[str]:
    if spacy_en is None:
        _, spacy_en = load_spacy_models()
    return [tok.text.lower() for tok in spacy_en(text)]


class Vocab:
    def __init__(self, stoi: dict, itos: dict) -> None:
        self.stoi = stoi
        self.itos = itos

    def lookup_token(self, idx: int) -> str:
        return self.itos.get(idx, "<unk>")

    def lookup_indices(self, tokens: Sequence[str]) -> List[int]:
        unk_idx = self.stoi["<unk>"]
        return [self.stoi.get(token, unk_idx) for token in tokens]

    def __len__(self) -> int:
        return len(self.stoi)


class Multi30kDataset(Dataset):
    def __init__(
        self,
        split: str = "train",
        max_len: int = 100,
        src_vocab: Vocab = None,
        tgt_vocab: Vocab = None,
        min_freq: int = 2,
        spacy_de=None,
        spacy_en=None,
    ) -> None:
        super().__init__()
        self.split = split
        self.max_len = max_len
        self.min_freq = min_freq
        self.spacy_de, self.spacy_en = (spacy_de, spacy_en) if spacy_de and spacy_en else load_spacy_models()
        self.dataset = load_dataset(
            "bentrevett/multi30k",
            split=split,
        )

        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        if self.src_vocab is None or self.tgt_vocab is None:
            self.src_vocab, self.tgt_vocab = self.build_vocab()
        self.data = self.process_data()

    def _extract_pair(self, example: dict) -> Tuple[str, str]:
        if "de" in example and "en" in example:
            return example["de"], example["en"]
        if "translation" in example:
            translation = example["translation"]
            return translation["de"], translation["en"]
        raise KeyError("Could not find German/English sentence pair in example.")

    def build_vocab(self) -> Tuple[Vocab, Vocab]:
        src_counter = Counter()
        tgt_counter = Counter()
        for example in self.dataset:
            src_text, tgt_text = self._extract_pair(example)
            src_counter.update(tokenize_de(src_text, self.spacy_de))
            tgt_counter.update(tokenize_en(tgt_text, self.spacy_en))

        def make_vocab(counter: Counter) -> Vocab:
            stoi = {token: idx for idx, token in enumerate(SPECIAL_TOKENS)}
            itos = {idx: token for idx, token in enumerate(SPECIAL_TOKENS)}
            next_idx = len(SPECIAL_TOKENS)
            for token, freq in sorted(counter.items()):
                if freq >= self.min_freq and token not in stoi:
                    stoi[token] = next_idx
                    itos[next_idx] = token
                    next_idx += 1
            return Vocab(stoi=stoi, itos=itos)

        return make_vocab(src_counter), make_vocab(tgt_counter)

    def process_data(self) -> List[Tuple[torch.Tensor, torch.Tensor]]:
        processed = []
        for example in self.dataset:
            src_text, tgt_text = self._extract_pair(example)
            src_tokens = ["<sos>"] + tokenize_de(src_text, self.spacy_de)[: self.max_len - 2] + ["<eos>"]
            tgt_tokens = ["<sos>"] + tokenize_en(tgt_text, self.spacy_en)[: self.max_len - 2] + ["<eos>"]
            src_ids = torch.tensor(self.src_vocab.lookup_indices(src_tokens), dtype=torch.long)
            tgt_ids = torch.tensor(self.tgt_vocab.lookup_indices(tgt_tokens), dtype=torch.long)
            processed.append((src_ids, tgt_ids))
        return processed

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.data[idx]


def collate_fn(batch: Iterable[Tuple[torch.Tensor, torch.Tensor]], pad_idx: int = 1):
    src_batch, tgt_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, batch_first=True, padding_value=pad_idx)
    tgt_batch = pad_sequence(tgt_batch, batch_first=True, padding_value=pad_idx)
    return src_batch, tgt_batch


def get_dataloaders(batch_size: int = 128, max_len: int = 100):
    spacy_de, spacy_en = load_spacy_models()
    train_dataset = Multi30kDataset(
        split="train",
        max_len=max_len,
        spacy_de=spacy_de,
        spacy_en=spacy_en,
    )
    val_dataset = Multi30kDataset(
        split="validation",
        max_len=max_len,
        src_vocab=train_dataset.src_vocab,
        tgt_vocab=train_dataset.tgt_vocab,
        spacy_de=spacy_de,
        spacy_en=spacy_en,
    )
    test_dataset = Multi30kDataset(
        split="test",
        max_len=max_len,
        src_vocab=train_dataset.src_vocab,
        tgt_vocab=train_dataset.tgt_vocab,
        spacy_de=spacy_de,
        spacy_en=spacy_en,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=lambda batch: collate_fn(batch, pad_idx=1),
        num_workers=2,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=lambda batch: collate_fn(batch, pad_idx=1),
        num_workers=0,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=lambda batch: collate_fn(batch, pad_idx=1),
        num_workers=0,
    )
    return (
        train_loader,
        val_loader,
        test_loader,
        train_dataset.src_vocab,
        train_dataset.tgt_vocab,
        spacy_de,
    )
'''
TRAIN_PY = r'''"""
train.py - Training, evaluation, checkpointing, and experiments.
"""

import glob
import os
from typing import Optional

import evaluate
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from dataset import get_dataloaders
from lr_scheduler import NoamScheduler
from model import Transformer, make_src_mask, make_tgt_mask


class LabelSmoothingLoss(nn.Module):
    def __init__(self, vocab_size: int, pad_idx: int, smoothing: float = 0.1) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.pad_idx = pad_idx
        self.smoothing = smoothing

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        log_probs = F.log_softmax(logits, dim=-1)
        smooth_value = self.smoothing / max(1, self.vocab_size - 1)
        smooth_dist = torch.full_like(log_probs, smooth_value)
        confidence = 1.0 - self.smoothing
        smooth_dist.scatter_(1, target.unsqueeze(1), confidence)
        pad_mask = target == self.pad_idx
        smooth_dist[pad_mask] = 0.0
        token_loss = -(smooth_dist * log_probs).sum(dim=-1)
        valid_mask = ~pad_mask
        if valid_mask.any():
            return token_loss[valid_mask].mean()
        return token_loss.mean() * 0.0


def _detokenize_text(text: str) -> str:
    replacements = {
        " .": ".",
        " ,": ",",
        " !": "!",
        " ?": "?",
        " :": ":",
        " ;": ";",
        " n't": "n't",
        " 's": "'s",
        " 're": "'re",
        " 've": "'ve",
        " 'm": "'m",
        " 'll": "'ll",
        " 'd": "'d",
        "( ": "(",
        " )": ")",
    }
    for src, dst in replacements.items():
        text = text.replace(src, dst)
    return text.strip()


def run_epoch(
    data_iter,
    model: Transformer,
    loss_fn: nn.Module,
    optimizer: Optional[torch.optim.Optimizer],
    scheduler=None,
    epoch_num: int = 0,
    is_train: bool = True,
    device: str = "cpu",
) -> float:
    model.train(is_train)
    total_loss = 0.0
    total_steps = 0
    progress = tqdm(data_iter, desc=f"{'Train' if is_train else 'Eval'} Epoch {epoch_num + 1}", leave=False)

    if not hasattr(run_epoch, "global_step"):
        run_epoch.global_step = 0

    for src, tgt in progress:
        src = src.to(device)
        tgt = tgt.to(device)
        tgt_input = tgt[:, :-1]
        tgt_gold = tgt[:, 1:]

        src_mask = make_src_mask(src, pad_idx=model.pad_idx).to(device)
        tgt_mask = make_tgt_mask(tgt_input, pad_idx=model.pad_idx).to(device)

        if is_train and optimizer is not None:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            logits = model(src, tgt_input, src_mask, tgt_mask)
            logits_flat = logits.reshape(-1, logits.size(-1))
            tgt_flat = tgt_gold.reshape(-1)
            loss = loss_fn(logits_flat, tgt_flat)

            if is_train and optimizer is not None:
                loss.backward()
                grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()
                run_epoch.global_step += 1

                if wandb.run is not None:
                    current_lr = (
                        scheduler.get_last_lr()[0]
                        if scheduler is not None
                        else optimizer.param_groups[0]["lr"]
                    )
                    with torch.no_grad():
                        probs = F.softmax(logits_flat, dim=-1)
                        valid_mask = tgt_flat != model.pad_idx
                        if valid_mask.any():
                            correct_prob = probs.gather(1, tgt_flat.unsqueeze(1)).squeeze(1)[valid_mask].mean().item()
                        else:
                            correct_prob = 0.0

                    log_payload = {
                        "train_loss": float(loss.item()),
                        "lr": float(current_lr),
                        "step": run_epoch.global_step,
                        "grad_clip_norm": float(grad_norm),
                        "pred_confidence": float(correct_prob),
                    }
                    if run_epoch.global_step <= 1000:
                        for name, param in model.named_parameters():
                            if ("W_q" in name or "W_k" in name) and param.grad is not None:
                                log_payload[f"grad_norm/{name}"] = float(param.grad.norm().item())
                    if run_epoch.global_step % 50 == 0:
                        wandb.log(log_payload)

        total_loss += float(loss.item())
        total_steps += 1
        progress.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / max(1, total_steps)


def greedy_decode(
    model: Transformer,
    src: torch.Tensor,
    src_mask: torch.Tensor,
    max_len: int,
    start_symbol: int,
    end_symbol: int,
    device: str = "cpu",
) -> torch.Tensor:
    model.eval()
    src = src.to(device)
    src_mask = src_mask.to(device)
    memory = model.encode(src, src_mask)
    ys = torch.tensor([[start_symbol]], dtype=torch.long, device=device)

    for _ in range(max_len - 1):
        tgt_mask = make_tgt_mask(ys, pad_idx=model.pad_idx).to(device)
        out = model.decode(memory, src_mask, ys, tgt_mask)
        next_word = torch.argmax(out[:, -1, :], dim=-1).item()
        ys = torch.cat(
            [ys, torch.tensor([[next_word]], dtype=torch.long, device=device)],
            dim=1,
        )
        if next_word == end_symbol:
            break
    return ys


def _ids_to_sentence(token_ids, vocab) -> str:
    specials = {"<sos>", "<eos>", "<pad>"}
    tokens = []
    for idx in token_ids:
        token = vocab.lookup_token(int(idx))
        if token in specials:
            continue
        tokens.append(token)
    return _detokenize_text(" ".join(tokens))


def evaluate_bleu(
    model: Transformer,
    test_dataloader: DataLoader,
    tgt_vocab,
    device: str = "cpu",
    max_len: int = 100,
) -> float:
    metric = evaluate.load("sacrebleu")
    predictions = []
    references = []

    model.eval()
    with torch.no_grad():
        for src_batch, tgt_batch in tqdm(test_dataloader, desc="BLEU", leave=False):
            src_batch = src_batch.to(device)
            tgt_batch = tgt_batch.to(device)
            for i in range(src_batch.size(0)):
                src = src_batch[i : i + 1]
                tgt = tgt_batch[i]
                src_mask = make_src_mask(src, pad_idx=model.pad_idx).to(device)
                pred_ids = greedy_decode(
                    model,
                    src,
                    src_mask,
                    max_len=max_len,
                    start_symbol=tgt_vocab.stoi["<sos>"],
                    end_symbol=tgt_vocab.stoi["<eos>"],
                    device=device,
                ).squeeze(0)
                predictions.append(_ids_to_sentence(pred_ids.tolist(), tgt_vocab))
                references.append([_ids_to_sentence(tgt.tolist(), tgt_vocab)])

    result = metric.compute(predictions=predictions, references=references, force=True)
    return float(result["score"])


def save_checkpoint(
    model: Transformer,
    optimizer: torch.optim.Optimizer,
    scheduler,
    epoch: int,
    path: str = "checkpoint.pt",
) -> None:
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "vocab_metadata": {
            "src_stoi": getattr(model, "src_stoi", {}),
            "tgt_stoi": getattr(model, "tgt_stoi", {}),
            "src_itos": getattr(model, "src_itos", {}),
            "tgt_itos": getattr(model, "tgt_itos", {}),
        },
        "model_config": {
            "src_vocab_size": model.src_vocab_size,
            "tgt_vocab_size": model.tgt_vocab_size,
            "d_model": model.d_model,
            "N": model.N,
            "num_heads": model.num_heads,
            "d_ff": model.d_ff,
            "dropout": model.dropout_rate,
            "pad_idx": model.pad_idx,
            "use_learned_positional_encoding": model.use_learned_positional_encoding,
            "attention_use_scaling": model.attention_use_scaling,
            "max_len": model.max_len,
        },
    }
    torch.save(checkpoint, path)


def load_checkpoint(
    path: str,
    model: Transformer,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler=None,
) -> int:
    checkpoint = torch.load(path, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"])
    vocab_metadata = checkpoint.get("vocab_metadata", {})
    model.src_stoi = vocab_metadata.get("src_stoi", getattr(model, "src_stoi", {}))
    model.tgt_stoi = vocab_metadata.get("tgt_stoi", getattr(model, "tgt_stoi", {}))
    model.src_itos = vocab_metadata.get("src_itos", getattr(model, "src_itos", {}))
    model.tgt_itos = vocab_metadata.get("tgt_itos", getattr(model, "tgt_itos", {}))
    if optimizer is not None and checkpoint.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    return int(checkpoint["epoch"])


def _maybe_upload_to_drive(local_path: str) -> None:
    folder_id = os.environ.get("GDRIVE_FOLDER_ID", "")
    if not folder_id:
        return
    try:
        import __main__

        if hasattr(__main__, "upload_to_gdrive"):
            __main__.upload_to_gdrive(local_path, folder_id)
    except Exception as exc:
        print(f"Drive upload skipped: {exc}")


def _sanitize_run_name(name: str) -> str:
    return name.replace(" ", "_").replace("/", "_").lower()


def _run_single_experiment(
    run_name: str,
    base_config: dict,
    train_loader,
    val_loader,
    test_loader,
    src_vocab,
    tgt_vocab,
    src_tokenizer,
    device: str,
    use_noam: bool = True,
    fixed_lr: float = 1.0,
    use_scaling: bool = True,
    use_learned_positional_encoding: bool = False,
    smoothing: float = 0.1,
    resume_main_run: bool = False,
    save_primary_best: bool = False,
    num_epochs: Optional[int] = None,
) -> float:
    config = dict(base_config)
    if num_epochs is not None:
        config["num_epochs"] = num_epochs
    run_id_base = os.environ.get("WANDB_RUN_ID", "da6401-a3-run")
    run_id = run_id_base if resume_main_run else f"{run_id_base}-{_sanitize_run_name(run_name)}"
    wandb_project = os.environ.get("WANDB_PROJECT", "da6401-a3-submission")
    wandb.init(
        project=wandb_project,
        id=run_id,
        resume="allow",
        name=run_name,
        config={
            **config,
            "use_noam": use_noam,
            "fixed_lr": fixed_lr if not use_noam else None,
            "attention_scaling": use_scaling,
            "use_learned_positional_encoding": use_learned_positional_encoding,
            "label_smoothing": smoothing,
        },
    )

    model = Transformer(
        src_vocab_size=len(src_vocab),
        tgt_vocab_size=len(tgt_vocab),
        d_model=config["d_model"],
        N=config["N"],
        num_heads=config["num_heads"],
        d_ff=config["d_ff"],
        dropout=config["dropout"],
        pad_idx=1,
        src_itos=src_vocab.itos,
        tgt_itos=tgt_vocab.itos,
        tgt_stoi=tgt_vocab.stoi,
        src_stoi=src_vocab.stoi,
        src_tokenizer=src_tokenizer,
        use_learned_positional_encoding=use_learned_positional_encoding,
        attention_use_scaling=use_scaling,
        max_len=config["max_len"],
    ).to(device)

    optimizer_lr = 1.0 if use_noam else fixed_lr
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=optimizer_lr,
        betas=(0.9, 0.98),
        eps=1e-9,
    )
    scheduler = (
        NoamScheduler(
            optimizer,
            d_model=config["d_model"],
            warmup_steps=config["warmup_steps"],
        )
        if use_noam
        else None
    )
    loss_fn = LabelSmoothingLoss(len(tgt_vocab), pad_idx=1, smoothing=smoothing)

    safe_name = _sanitize_run_name(run_name)
    checkpoint_pattern = f"checkpoint_{safe_name}_epoch_*.pt"
    start_epoch = 0
    checkpoints = sorted(glob.glob(checkpoint_pattern))
    if checkpoints:
        latest = checkpoints[-1]
        start_epoch = load_checkpoint(latest, model, optimizer, scheduler) + 1
        print(f"Resumed from {latest}, starting at epoch {start_epoch + 1}")

    best_bleu = -1.0
    best_path = f"best_model_{safe_name}.pt"

    for epoch in range(start_epoch, config["num_epochs"]):
        if device.startswith("cuda"):
            torch.cuda.empty_cache()

        train_loss = run_epoch(
            train_loader,
            model,
            loss_fn,
            optimizer,
            scheduler=scheduler,
            epoch_num=epoch,
            is_train=True,
            device=device,
        )
        val_loss = run_epoch(
            val_loader,
            model,
            loss_fn,
            None,
            scheduler=None,
            epoch_num=epoch,
            is_train=False,
            device=device,
        )
        val_bleu = evaluate_bleu(
            model,
            val_loader,
            tgt_vocab=tgt_vocab,
            device=device,
            max_len=config["max_len"],
        )

        wandb.log(
            {
                "epoch": epoch + 1,
                "train_epoch_loss": train_loss,
                "val_loss": val_loss,
                "val_bleu": val_bleu,
            }
        )

        epoch_ckpt = f"checkpoint_{safe_name}_epoch_{epoch + 1:03d}.pt"
        save_checkpoint(model, optimizer, scheduler, epoch, epoch_ckpt)

        if val_bleu > best_bleu:
            best_bleu = val_bleu
            save_checkpoint(model, optimizer, scheduler, epoch, best_path)
            if save_primary_best:
                save_checkpoint(model, optimizer, scheduler, epoch, "best_model.pt")

    load_checkpoint(best_path, model)
    if save_primary_best and os.path.exists("best_model.pt"):
        _maybe_upload_to_drive("best_model.pt")
    test_bleu = evaluate_bleu(
        model,
        test_loader,
        tgt_vocab=tgt_vocab,
        device=device,
        max_len=config["max_len"],
    )
    wandb.log({"best_val_bleu": best_bleu, "final_test_bleu": test_bleu})
    wandb.finish()
    return test_bleu


def run_training_experiment() -> None:
    CONFIG = {
        "d_model": 256,
        "N": 3,
        "num_heads": 8,
        "d_ff": 512,
        "dropout": 0.1,
        "batch_size": 128,
        "num_epochs": 25,
        "warmup_steps": 4000,
        "label_smoothing": 0.1,
        "max_len": 100,
        "clip": 1.0,
    }

    device = "cuda" if torch.cuda.is_available() else "cpu"
    train_loader, val_loader, test_loader, src_vocab, tgt_vocab, spacy_de = get_dataloaders(
        batch_size=CONFIG["batch_size"],
        max_len=CONFIG["max_len"],
    )

    baseline_bleu = _run_single_experiment(
        run_name="noam-baseline",
        base_config=CONFIG,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        src_vocab=src_vocab,
        tgt_vocab=tgt_vocab,
        src_tokenizer=spacy_de,
        device=device,
        use_noam=True,
        use_scaling=True,
        use_learned_positional_encoding=False,
        smoothing=CONFIG["label_smoothing"],
        resume_main_run=True,
        save_primary_best=True,
    )

    ablation_epochs = int(os.environ.get("ABLATION_EPOCHS", "10"))
    _run_single_experiment(
        run_name="fixed-lr-1e4",
        base_config=CONFIG,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        src_vocab=src_vocab,
        tgt_vocab=tgt_vocab,
        src_tokenizer=spacy_de,
        device=device,
        use_noam=False,
        fixed_lr=1e-4,
        smoothing=CONFIG["label_smoothing"],
        num_epochs=ablation_epochs,
    )
    _run_single_experiment(
        run_name="no-scaling",
        base_config=CONFIG,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        src_vocab=src_vocab,
        tgt_vocab=tgt_vocab,
        src_tokenizer=spacy_de,
        device=device,
        use_noam=True,
        use_scaling=False,
        smoothing=CONFIG["label_smoothing"],
        num_epochs=ablation_epochs,
    )
    _run_single_experiment(
        run_name="learned-positional-encoding",
        base_config=CONFIG,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        src_vocab=src_vocab,
        tgt_vocab=tgt_vocab,
        src_tokenizer=spacy_de,
        device=device,
        use_noam=True,
        use_scaling=True,
        use_learned_positional_encoding=True,
        smoothing=CONFIG["label_smoothing"],
        num_epochs=ablation_epochs,
    )
    _run_single_experiment(
        run_name="no-label-smoothing",
        base_config=CONFIG,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        src_vocab=src_vocab,
        tgt_vocab=tgt_vocab,
        src_tokenizer=spacy_de,
        device=device,
        use_noam=True,
        use_scaling=True,
        smoothing=0.0,
        num_epochs=ablation_epochs,
    )

    print(f"Baseline test BLEU: {baseline_bleu:.2f}")


if __name__ == "__main__":
    run_training_experiment()
'''

for fname, content in [
    ('model.py', MODEL_PY),
    ('lr_scheduler.py', LR_PY),
    ('dataset.py', DATASET_PY),
    ('train.py', TRAIN_PY),
]:
    with open(fname, 'w') as f:
        f.write(content)

print('Source files written.')


In [ ]:
import os
import sys

sys.path.insert(0, '/kaggle/working/da6401_a3')
os.environ['ABLATION_EPOCHS'] = '10'

from train import run_training_experiment
run_training_experiment()


In [ ]:
import torch
import wandb
from dataset import get_dataloaders
from model import Transformer
from train import evaluate_bleu, load_checkpoint

device = 'cuda' if torch.cuda.is_available() else 'cpu'
_, _, test_loader, src_vocab, tgt_vocab, _ = get_dataloaders(batch_size=64, max_len=100)

checkpoint = torch.load('best_model.pt', map_location=device)
cfg = checkpoint['model_config']
model = Transformer(**cfg).to(device)
load_checkpoint('best_model.pt', model)
model.eval()

bleu = evaluate_bleu(model, test_loader, tgt_vocab, device=device, max_len=100)
print(f'\nFinal Test BLEU: {bleu:.2f}')

wandb.init(project=WANDB_PROJECT, id=WANDB_RUN_ID, resume='allow')
wandb.log({'final_test_bleu_recheck': bleu})
wandb.finish()


In [ ]:
import matplotlib.pyplot as plt
import torch
import wandb
from dataset import load_spacy_models, tokenize_de
from model import make_src_mask

wandb.init(project=WANDB_PROJECT, id=WANDB_RUN_ID, resume='allow')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
spacy_de, _ = load_spacy_models()
sample_de = 'Ein Mann sitzt auf einer Bank.'

tokens = ['<sos>'] + tokenize_de(sample_de, spacy_de) + ['<eos>']
src_ids = [src_vocab.stoi.get(t, 0) for t in tokens]
src_tensor = torch.tensor(src_ids).unsqueeze(0).to(device)
src_mask = make_src_mask(src_tensor, pad_idx=1).to(device)

with torch.no_grad():
    _ = model.encode(src_tensor, src_mask)

attn_weights = model.encoder.layers[-1].self_attn.attn_weights
num_heads = attn_weights.size(1)
rows = 2
cols = (num_heads + 1) // 2
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 8))
axes = axes.flatten()

for h in range(num_heads):
    w = attn_weights[0, h].cpu().numpy()
    ax = axes[h]
    im = ax.imshow(w, cmap='viridis', vmin=0, vmax=w.max() if w.max() > 0 else 1.0)
    ax.set_title(f'Head {h + 1}')
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=90, fontsize=8)
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=8)
    plt.colorbar(im, ax=ax)

for ax in axes[num_heads:]:
    ax.axis('off')

plt.suptitle(f'Encoder Last-Layer Attention Maps\n"{sample_de}"')
plt.tight_layout()
plt.savefig('attention_maps.png', dpi=150, bbox_inches='tight')
wandb.log({'attention_maps': wandb.Image('attention_maps.png')})
wandb.finish()
print('Attention maps logged to W&B.')


In [ ]:
import json
import os
import shutil

final_local_name = f'best_model_{WANDB_RUN_ID}.pt'
shutil.copy('best_model.pt', '/kaggle/working/best_model.pt')
shutil.copy('best_model.pt', f'/kaggle/working/{final_local_name}')

manifest = {
    'wandb_project': WANDB_PROJECT,
    'wandb_run_id': WANDB_RUN_ID,
    'local_best_model': '/kaggle/working/best_model.pt',
    'local_versioned_model': f'/kaggle/working/{final_local_name}',
    'gdrive_folder_id': GDRIVE_FOLDER_ID,
}
with open('/kaggle/working/submission_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

upload_to_gdrive('best_model.pt', GDRIVE_FOLDER_ID, remote_name=f'da6401_a3_{WANDB_RUN_ID}_best_model.pt')
upload_to_gdrive('/kaggle/working/submission_manifest.json', GDRIVE_FOLDER_ID, remote_name=f'da6401_a3_{WANDB_RUN_ID}_manifest.json')

print('Saved /kaggle/working/best_model.pt')
print(f'Saved /kaggle/working/{final_local_name}')
print('Saved /kaggle/working/submission_manifest.json')
print('Notebook run complete.')
